#EXPLORATORY DATA ANALYSIS (EDA)

#THE GAOL OF THIS EDA

We will answer the following Clinical and non clinical questions:


- Which biomarkers are most associated with mortality?
- How do survivors differ from non-survivors?
- Which physiological systems fail in sepsis?
- Which variables behave abnormally in critical patients?
- Which features are correlated?
- Which variables are skewed?
- Which variables are noisy?
- Which variables are missing frequently?
- Which features are useful?
- Which features are redundant?
- Which features need transformation?

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from scipy.stats import mannwhitneyu

from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader
from sklearn.svm import SVC
import itertools

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (

    accuracy_score,

    precision_score,
    recall_score,
    f1_score,

    roc_auc_score,
    roc_curve,

    precision_recall_curve,
    average_precision_score,

    confusion_matrix,
    classification_report,

    balanced_accuracy_score,

    ConfusionMatrixDisplay
)

from sklearn.calibration import (
    calibration_curve
)

from imblearn.over_sampling import SMOTE

from sklearn.pipeline import Pipeline


from sklearn.preprocessing import (
    RobustScaler,
    StandardScaler
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    TensorDataset
)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


import seaborn as sns
%matplotlib inline

#PHASE A: DATASET OVERVIEW

In [ ]:
df = pd.read_csv("/content/sepsis data.csv")

In [ ]:
df.head()

In [ ]:
#lets see the shape of the data
print(df.shape)

In [ ]:
#lets display all the features
df.columns

In [ ]:
#lets see the amount of non-null value for each feature
df.info()

#PHASE B: TARGET VARIABLE ANALYSIS

In [ ]:
df["TARGET_MORTALITY"].value_counts()

In [ ]:

sns.set_style("whitegrid")
plt.figure(figsize=(8,6))
ax = sns.countplot(
    x="TARGET_MORTALITY",
    data=df,
    palette=["#4CAF50", "#E53935"]
)
plt.title("Distribution of Mortality Outcomes in Sepsis Patients",
          fontsize=16,
          fontweight='bold')

plt.xlabel("Mortality Outcome", fontsize=13)
plt.ylabel("Number of Patients", fontsize=13)

ax.set_xticklabels(["Survived", "Died"])
total = len(df)

for p in ax.patches:
    count = int(p.get_height())
    percentage = 100 * count / total

    ax.annotate(
        f'{count}\n({percentage:.1f}%)',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

sns.despine()

plt.show()

The mortality distribution shows that the dataset is moderately imbalanced, with more surviving patients than deceased patients. Approximately one-third of ICU sepsis patients in the cohort died during hospitalization, highlighting the severity of sepsis in critical care settings. This class imbalance is important because relying solely on accuracy during model evaluation could be misleading. Therefore, metrics such as recall, F1-score, ROC-AUC, sensitivity, and specificity will be prioritized during model evaluation

#PHASE C : MISSING VALUE ANALYSIS

In [ ]:
missing = df.isnull().mean() * 100
missing = missing.sort_values(ascending=False)

In [ ]:

missing = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_top20 = missing.head(20)
sns.set_style("whitegrid")

plt.figure(figsize=(14,6))
colors = sns.color_palette("Reds_r", len(missing_top20))
ax = sns.barplot(
    x=missing_top20.index,
    y=missing_top20.values,
    palette=colors
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel("Percentage Missing (%)", fontsize=13)
plt.xlabel("Features", fontsize=13)

plt.title(
    "Top 20 Features with Missing Values",
    fontsize=17,
    fontweight='bold'
)

for i, v in enumerate(missing_top20.values):
    ax.text(
        i,
        v + 0.5,
        f"{v:.1f}%",
        ha='center',
        fontsize=10,
        fontweight='bold'
    )

sns.despine()

plt.tight_layout()
plt.show()

The missing value analysis reveals that several laboratory biomarkers exhibit moderate missingness, particularly bilirubin, PaCO2, PaO2, pH, and lactate measurements. This pattern is clinically expected in ICU settings because these laboratory tests are typically ordered selectively based on patient severity and physician judgment rather than being routinely measured for every patient. with their missing percentages indicated, we will handle it clearly

#PHASE D: DISTRIBUTION ANALYSIS AND PRE-MODELING FEATURE IMPORTANCE EXPLORATION

The aim here is to visually check whether each feature separates survivors (0) from non-survivors (1). If the distributions are clearly different, the feature may be useful for the ML model. we will use KDE, histogramming and boxplot to see this.

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
TARGET = "TARGET_MORTALITY"
all_features = [
    'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'TARGET_MORTALITY', 'AGE',
    'GENDER', 'ETHNICITY', 'ADMISSION_TYPE', 'heartrate_min',
    'heartrate_max', 'heartrate_mean', 'sysbp_min', 'sysbp_max',
    'sysbp_mean', 'diasbp_min', 'diasbp_max', 'diasbp_mean', 'meanbp_min',
    'meanbp_max', 'meanbp_mean', 'resprate_min', 'resprate_max',
    'resprate_mean', 'temperature_min', 'temperature_max',
    'temperature_mean', 'spo2_min', 'spo2_max', 'spo2_mean', 'lactate_min',
    'lactate_max', 'lactate_mean', 'wbc_min', 'wbc_max', 'wbc_mean',
    'creatinine_min', 'creatinine_max', 'creatinine_mean', 'bilirubin_min',
    'bilirubin_max', 'bilirubin_mean', 'platelet_min', 'platelet_max',
    'platelet_mean', 'bun_min', 'bun_max', 'bun_mean', 'sodium_min',
    'sodium_max', 'sodium_mean', 'potassium_min', 'potassium_max',
    'potassium_mean', 'bicarbonate_min', 'bicarbonate_max',
    'bicarbonate_mean', 'ph_min', 'ph_max', 'ph_mean', 'pao2_min',
    'pao2_max', 'pao2_mean', 'paco2_min', 'paco2_max', 'paco2_mean',
    'glucose_min', 'glucose_max', 'glucose_mean', 'hemoglobin_min',
    'hemoglobin_max', 'hemoglobin_mean', 'urineoutput_24h',
    'vasopressor_24h', 'sofa_score'
]

# Features we should NOT plot as continuous biomarkers
excluded_features = [
    "SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "TARGET_MORTALITY",
    "GENDER", "ETHNICITY", "ADMISSION_TYPE"
]

numeric_features = [
    f for f in all_features
    if f not in excluded_features and f in df.columns
]

print("Number of numeric clinical features:", len(numeric_features))
print(numeric_features)

#1. KDE

In [ ]:
def plot_kde_features(df, features, start=0, end=6, target=TARGET):
    selected_features = features[start:end]

    for feature in selected_features:
        plt.figure(figsize=(9, 5))

        sns.kdeplot(
            data=df,
            x=feature,
            hue=target,
            fill=True,
            common_norm=False,
            alpha=0.35,
            linewidth=2,
            palette={0: "#2E7D32", 1: "#C62828"}
        )

        plt.title(
            f"KDE Distribution of {feature} by Mortality Outcome",
            fontsize=15,
            fontweight="bold"
        )

        plt.xlabel(feature, fontsize=12)
        plt.ylabel("Density", fontsize=12)

        plt.legend(
            title="Outcome",
            labels=["Died", "Survived"]
        )

        sns.despine()
        plt.tight_layout()
        plt.show()

In [ ]:
plot_kde_features(df, numeric_features, start=0, end=5)

In [ ]:



sns.set_theme(style="whitegrid", context="notebook")

def plot_kde_grid(
    df,
    features,
    start=0,
    end=24,
    target="TARGET_MORTALITY",
    cols=4
):

    selected_features = features[start:end]

    n_features = len(selected_features)
    rows = math.ceil(n_features / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(cols * 5, rows * 3.8)
    )

    axes = axes.flatten()

    for i, feature in enumerate(selected_features):

        sns.kdeplot(
            data=df,
            x=feature,
            hue=target,
            fill=True,
            common_norm=False,
            alpha=0.35,
            linewidth=1.8,
            palette={0: "#2E7D32", 1: "#C62828"},
            ax=axes[i]
        )

        axes[i].set_title(
            feature,
            fontsize=11,
            fontweight="bold"
        )

        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

        axes[i].legend_.remove()

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        "KDE Distribution of Features by Mortality Outcome",
        fontsize=20,
        fontweight="bold"
    )
    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        ["Survived", "Died"],
        loc="upper right",
        fontsize=12
    )

    plt.tight_layout(rect=[0, 0, 0.97, 0.96])

    plt.show()

In [ ]:
plot_kde_grid(
    df,
    numeric_features,
    start=0,
    end=12,
    cols=4
)

In [ ]:
plot_kde_grid(
    df,
    numeric_features,
    start=12,
    end=24,
    cols=4
)

In [ ]:
plot_kde_grid(
    df,
    numeric_features,
    start=24,
    end=36,
    cols=4
)

In [ ]:
plot_kde_grid(
    df,
    numeric_features,
    start=36,
    end=48,
    cols=4
)

In [ ]:
plot_kde_grid(
    df,
    numeric_features,
    start=48,
    end=67,
    cols=4
)

#Honestly, these KDE plots already reveal a LOT about:

- sepsis physiology,
- mortality mechanisms,
- biomarker importance,
- and likely predictive power for machine learning.

#this also clearly show

- distribution shifts,
- overlap,
- skewness,
- and separation between mortality groups.

overall, the KDE distributions show that several physiological and laboratory variables exhibit clear distributional differences between survivors and non-survivors. Features associated with organ dysfunction, hemodynamic instability, renal failure, metabolic dysregulation, and septic shock demonstrate stronger separation between mortality groups, suggesting high predictive relevance for mortality modeling.

#STRONGEST BIOMARKERS OBSERVED
some features showed a very strong capabilities like:
- sofa score
- lactate
- VASOPRESSOR USE
- Creatinine + BUN
- platelete
- BILIRUBIN
- respiratory variables
- BLOOD PRESSURE VARIABLES.

#and features with weak biomarkers
- sodium,
- potassium,
- hemoglobin,
- some temperature variables.


#lets proceed to see the actual strenth of this biomarkers

In [ ]:



# Cohen's d effect size


def cohens_d(x1, x2):

    x1 = x1.dropna()
    x2 = x2.dropna()

    n1, n2 = len(x1), len(x2)

    s1, s2 = np.var(x1, ddof=1), np.var(x2, ddof=1)

    pooled_std = np.sqrt(
        ((n1 - 1)*s1 + (n2 - 1)*s2) / (n1 + n2 - 2)
    )

    return (np.mean(x1) - np.mean(x2)) / pooled_std



# Biomarker strength analysis

results = []

survivors = df[df["TARGET_MORTALITY"] == 0]
nonsurvivors = df[df["TARGET_MORTALITY"] == 1]

for feature in numeric_features:

    try:

        s = survivors[feature].dropna()
        d = nonsurvivors[feature].dropna()


        stat, p = mannwhitneyu(s, d)

        effect = cohens_d(s, d)

        surv_mean = s.mean()
        death_mean = d.mean()

        results.append({
            "Feature": feature,
            "Survivor_Mean": surv_mean,
            "NonSurvivor_Mean": death_mean,
            "P_Value": p,
            "Effect_Size_Cohens_d": effect,
            "Absolute_Effect_Size": abs(effect)
        })

    except:
        pass

biomarker_table = pd.DataFrame(results)

biomarker_table = biomarker_table.sort_values(
    by="Absolute_Effect_Size",
    ascending=False
)

biomarker_table.reset_index(drop=True, inplace=True)

# Display top biomarkers
biomarker_table.head(20)

In [ ]:

top_features = biomarker_table.head(20)

plt.figure(figsize=(12,7))

ax = sns.barplot(
    data=top_features,
    y="Feature",
    x="Absolute_Effect_Size",
    palette="Reds_r"
)

plt.title(
    "Top Biomarkers Associated with Mortality",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Absolute Effect Size (Cohen's d)", fontsize=13)
plt.ylabel("Feature", fontsize=13)

for p in ax.patches:
    width = p.get_width()

    ax.annotate(
        f"{width:.2f}",
        (width, p.get_y() + p.get_height()/2),
        ha="left",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

sns.despine()

plt.tight_layout()
plt.show()

This biomarker strength table provides strong evidence that mortality in sepsis is closely associated with multiorgan dysfunction, impaired tissue perfusion, circulatory instability, respiratory/metabolic imbalance, and renal failure. Several biomarkers demonstrated substantial separation between survivors and non-survivors, suggesting high predictive relevance for mortality modeling.

#let us now confirm this with boxplot and histogram

#2. BOXPLOT
this will also dectect outliers in the features

In [ ]:


def plot_boxplot_grid(
    df,
    features,
    start=0,
    end=24,
    target="TARGET_MORTALITY",
    cols=4
):

    df[target] = df[target].astype(int)

    selected_features = features[start:end]

    n_features = len(selected_features)
    rows = math.ceil(n_features / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(cols * 5, rows * 4)
    )

    axes = axes.flatten()

    for i, feature in enumerate(selected_features):

        sns.boxplot(
            data=df,
            x=target,
            y=feature,
            hue=target,
            palette={0: "#2E7D32", 1: "#C62828"},
            legend=False,
            width=0.55,
            ax=axes[i]
        )

        axes[i].set_title(
            feature,
            fontsize=11,
            fontweight="bold"
        )

        axes[i].set_xticklabels(
            ["Survived", "Died"],
            fontsize=9
        )

        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        "Feature Distribution by Mortality Outcome (Boxplots)",
        fontsize=20,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    plt.show()

In [ ]:
plot_boxplot_grid(
    df,
    numeric_features,
    start=0,
    end=24,
    cols=4
)

#3. HISTOGRAM

In [ ]:
def plot_hist_grid(
    df,
    features,
    start=0,
    end=24,
    target="TARGET_MORTALITY",
    cols=4,
    bins=35
):

    selected_features = features[start:end]

    n_features = len(selected_features)
    rows = math.ceil(n_features / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(cols * 5, rows * 3.8)
    )

    axes = axes.flatten()

    for i, feature in enumerate(selected_features):

        sns.histplot(
            data=df,
            x=feature,
            hue=target,
            bins=bins,
            kde=True,
            stat="density",
            common_norm=False,
            alpha=0.4,
            palette={0: "#2E7D32", 1: "#C62828"},
            ax=axes[i]
        )

        axes[i].set_title(
            feature,
            fontsize=11,
            fontweight="bold"
        )

        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

        if axes[i].legend_:
            axes[i].legend_.remove()

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        "Histogram Distribution of Features by Mortality Outcome",
        fontsize=20,
        fontweight="bold"
    )

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        ["Survived", "Died"],
        loc="upper right",
        fontsize=12
    )

    plt.tight_layout(rect=[0, 0, 0.97, 0.96])

    plt.show()

In [ ]:
plot_hist_grid(
    df,
    numeric_features,
    start=0,
    end=24,
    cols=4
)

THIS INDEED CONFIRM THE SAME THINK LIKE EDA

#PHASE E:  OUTLIER ANALYSIS
Boxplot already reveal these outliers, here we will simply make a table summary for the outlier

In [ ]:


def outlier_summary(df, features):

    results = []

    for feature in features:

        try:

            series = df[feature].dropna()

            Q1 = series.quantile(0.25)
            Q3 = series.quantile(0.75)

            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            outliers = series[
                (series < lower) |
                (series > upper)
            ]

            results.append({

                "Feature": feature,

                "Min": series.min(),

                "Max": series.max(),

                "Lower_Bound": lower,

                "Upper_Bound": upper,

                "Num_Outliers": len(outliers),

                "Percent_Outliers":
                    100 * len(outliers) / len(series)

            })

        except:
            pass

    return pd.DataFrame(results)

In [ ]:
outlier_table = outlier_summary(df, numeric_features)

outlier_table.sort_values(
    by="Percent_Outliers",
    ascending=False
).head(20)

In [ ]:

top_outliers = outlier_table.sort_values(
    by="Percent_Outliers",
    ascending=False
).head(20)

sns.set_theme(style="whitegrid")

plt.figure(figsize=(14,8))

ax = sns.barplot(
    data=top_outliers,
    y="Feature",
    x="Percent_Outliers",
    palette="rocket"
)

plt.title(
    "Top Clinical Features with Highest Outlier Proportions",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel("Percentage of Outliers (%)", fontsize=14)
plt.ylabel("Clinical Features", fontsize=14)
for p in ax.patches:

    width = p.get_width()

    ax.annotate(
        f"{width:.1f}%",
        (width, p.get_y() + p.get_height()/2),
        ha="left",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

sns.despine()

plt.tight_layout()

plt.show()

#Outlier analysis revealed that some extreme physiological values likely reflected true severe pathology rather than erroneous measurements. Therefore, clinical plausibility was considered before deciding whether extreme observations should be retained, transformed, or removed.

#PHASE F: Correlation + Redundancy Analysis

the goal of phase G is to answer the questions:

1. Which features are strongly correlated?

3. Are some features redundant?

4. Should we later remove some features?

This affects preprocessing, model stability,
 interpretability.

Correlation does NOT automatically mean:
we should remove the feature. however, Sometimes max, min, mean capture DIFFERENT clinical behavior. so, careful analysis is needed.

In [ ]:
#lets build a corr matirx

corr_matrix = df[numeric_features].corr()

corr_matrix.head()

In [ ]:


plt.figure(figsize=(22,18))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink":0.7}
)

plt.title(
    "Correlation Heatmap of Clinical Features",
    fontsize=24,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

#lets extract higly correlated pair

In [ ]:

corr_abs = corr_matrix.abs()


upper = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)
high_corr = []

threshold = 0.85

for col in upper.columns:

    for row in upper.index:

        value = upper.loc[row, col]

        if pd.notnull(value) and value > threshold:

            high_corr.append({

                "Feature_1": row,
                "Feature_2": col,
                "Correlation": value

            })


high_corr_df = pd.DataFrame(high_corr)


high_corr_df = high_corr_df.sort_values(
    by="Correlation",
    ascending=False
)

high_corr_df.reset_index(drop=True, inplace=True)

high_corr_df.head(30)

#lets visulalise this

In [ ]:
top_corr = high_corr_df.head(20)

top_corr["Pair"] = (
    top_corr["Feature_1"]
    + " ↔ "
    + top_corr["Feature_2"]
)

plt.figure(figsize=(14,8))

ax = sns.barplot(
    data=top_corr,
    y="Pair",
    x="Correlation",
    palette="viridis"
)

plt.title(
    "Top Highly Correlated Clinical Feature Pairs",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel("Absolute Correlation", fontsize=14)
plt.ylabel("Feature Pair", fontsize=14)


for p in ax.patches:

    width = p.get_width()

    ax.annotate(
        f"{width:.2f}",
        (width, p.get_y() + p.get_height()/2),
        ha="left",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

sns.despine()

plt.tight_layout()

plt.show()

The correlation analysis revealed several extremely strong relationships between clinical variables, particularly among the engineered minimum, maximum, and mean versions of the same biomarker. This is expected because these aggregated statistics originate from the same underlying physiological measurements collected during the ICU stay.

These strong correlations suggest the presence of feature redundancy and potential multicollinearity within the dataset. However, this does not automatically imply that correlated features should be removed. In clinical time-series aggregation,

minimum values may capture physiological collapse,
maximum values may capture acute deterioration,
mean values may reflect sustained dysfunction over time.

Therefore, although statistically redundant, these variables may still contain complementary clinical information relevant for mortality prediction.

This analysis also confirms that the engineered features preserve physiologically coherent relationships, indicating that the SQL extraction and aggregation pipeline successfully captured clinically meaningful patient dynamics rather than random noise.

#PART 2: DATA PRE-PROCESSING



Identifiers were not used as predictors because they do not contain clinical meaning and may cause leakage or memorization.

#STEP 1: IMPOSSIBLE VALUE TREATMENT
detecting values that are biologically impossible or extremely unrealistic and handling them properly before training the model is an important concern. these values can mislead the model

In [ ]:


def impossible_value_treatment(
    df,
    target_col="TARGET_MORTALITY",
    id_cols=["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID"],
    categorical_cols=["GENDER", "ETHNICITY", "ADMISSION_TYPE"]
):

    data = df.copy()

    data = data.drop(
        columns=[col for col in id_cols if col in data.columns]
    )

  # Separate target
    y = data[target_col]

    X = data.drop(columns=[target_col])

     # Encode categorical variables

    categorical_cols = [
        col for col in categorical_cols
        if col in X.columns
    ]

    X = pd.get_dummies(
        X,
        columns=categorical_cols,
        drop_first=True
    )

    # Physiological validity rules
    physiological_rules = {

        # Heart rate
        "heartrate_min": (20, 250),
        "heartrate_max": (20, 250),
        "heartrate_mean": (20, 250),

        # Blood pressure
        "sysbp_min": (30, 300),
        "sysbp_max": (30, 300),
        "sysbp_mean": (30, 300),

        "diasbp_min": (20, 200),
        "diasbp_max": (20, 200),
        "diasbp_mean": (20, 200),

        "meanbp_min": (20, 250),
        "meanbp_max": (20, 250),
        "meanbp_mean": (20, 250),

        # Respiratory rate
        "resprate_min": (1, 80),
        "resprate_max": (1, 80),
        "resprate_mean": (1, 80),

        # Temperature
        "temperature_min": (25, 45),
        "temperature_max": (25, 45),
        "temperature_mean": (25, 45),

        # Oxygen saturation
        "spo2_min": (50, 100),
        "spo2_max": (50, 100),
        "spo2_mean": (50, 100),

        # Lactate
        "lactate_min": (0, 30),
        "lactate_max": (0, 30),
        "lactate_mean": (0, 30),

        # White blood cells
        "wbc_min": (0, 500),
        "wbc_max": (0, 500),
        "wbc_mean": (0, 500),

        # Creatinine
        "creatinine_min": (0, 30),
        "creatinine_max": (0, 30),
        "creatinine_mean": (0, 30),

        # Bilirubin
        "bilirubin_min": (0, 80),
        "bilirubin_max": (0, 80),
        "bilirubin_mean": (0, 80),

        # Platelets
        "platelet_min": (1, 2000),
        "platelet_max": (1, 2000),
        "platelet_mean": (1, 2000),

        # BUN
        "bun_min": (0, 300),
        "bun_max": (0, 300),
        "bun_mean": (0, 300),

        # Sodium
        "sodium_min": (100, 180),
        "sodium_max": (100, 180),
        "sodium_mean": (100, 180),

        # Potassium
        "potassium_min": (1, 10),
        "potassium_max": (1, 10),
        "potassium_mean": (1, 10),

        # Bicarbonate
        "bicarbonate_min": (0, 60),
        "bicarbonate_max": (0, 60),
        "bicarbonate_mean": (0, 60),

        # pH
        "ph_min": (6.8, 7.8),
        "ph_max": (6.8, 7.8),
        "ph_mean": (6.8, 7.8),

        # PaO2
        "pao2_min": (0, 700),
        "pao2_max": (0, 700),
        "pao2_mean": (0, 700),

        # PaCO2
        "paco2_min": (10, 200),
        "paco2_max": (10, 200),
        "paco2_mean": (10, 200),

        # Glucose
        "glucose_min": (0, 2000),
        "glucose_max": (0, 2000),
        "glucose_mean": (0, 2000),

        # Hemoglobin
        "hemoglobin_min": (0, 25),
        "hemoglobin_max": (0, 25),
        "hemoglobin_mean": (0, 25),

        # Urine output
        "urineoutput_24h": (0, 50000),

        # Vasopressor
        "vasopressor_24h": (0, 1),

        # SOFA
        "sofa_score": (0, 24)
    }


    # we will Convert impossible values to NaN

    impossible_summary = []

    for col, (low, high) in physiological_rules.items():

        if col in X.columns:

            invalid_mask = (
                (X[col] < low) |
                (X[col] > high)
            )

            num_invalid = invalid_mask.sum()

            percent_invalid = (
                num_invalid / len(X)
            ) * 100

            # Replace impossible values with NaN
            X.loc[invalid_mask, col] = np.nan

            impossible_summary.append({

                "Feature": col,
                "Num_Invalid": num_invalid,
                "Percent_Invalid": percent_invalid

            })


    processed_df = X.copy()

    processed_df[target_col] = y.values

    impossible_summary_df = pd.DataFrame(
        impossible_summary
    ).sort_values(
        by="Percent_Invalid",
        ascending=False
    )

    return processed_df, impossible_summary_df

In [ ]:
processed_df, impossible_summary_df = impossible_value_treatment(df)

In [ ]:
processed_df

In [ ]:
impossible_summary_df

In [ ]:
processed_df.isnull().sum().sort_values(ascending=False)

The physiological validity analysis revealed that several clinical variables contained biologically implausible values, most likely originating from ICU monitoring artifacts, recording errors, or sensor failures rather than true patient physiology.

The largest proportions of invalid measurements were observed in:

diastolic blood pressure minimum ,
respiratory rate minimum ,
systolic blood pressure minimum ,
and mean arterial pressure minimum ,and much more.

#step 1. HANDLING MISSING VALUES

In [ ]:
# Converting boolean columns to integers
bool_cols = processed_df.select_dtypes(include="bool").columns

processed_df[bool_cols] = processed_df[bool_cols].astype(int)

In [ ]:
processed_df.head()

In [ ]:
missing = processed_df.isnull().mean() * 100
missing = missing.sort_values(ascending=False)

In [ ]:

missing = (processed_df.isnull().mean() * 100).sort_values(ascending=False)

missing_top20 = missing.head(20)
sns.set_style("whitegrid")

plt.figure(figsize=(14,6))
colors = sns.color_palette("Reds_r", len(missing_top20))
ax = sns.barplot(
    x=missing_top20.index,
    y=missing_top20.values,
    palette=colors
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.ylabel("Percentage Missing (%)", fontsize=13)
plt.xlabel("Features", fontsize=13)

plt.title(
    "Top 20 Features with Missing Values",
    fontsize=17,
    fontweight='bold'
)

for i, v in enumerate(missing_top20.values):
    ax.text(
        i,
        v + 0.5,
        f"{v:.1f}%",
        ha='center',
        fontsize=10,
        fontweight='bold'
    )

sns.despine()

plt.tight_layout()
plt.show()

We will drop the features with more than 50% of the missing values. However, from the missing value visualisation we had, the highest % of values missing for a feature was 42%. this is not enough  tho drop them. we will therefore not drop any features. However, we will replace it with the mean and for catigorical features, we will replace it with the mode.

In [ ]:

def preprocess_missing_values(
    df,
    target_col="TARGET_MORTALITY"
):

    data = df.copy()

    data[target_col] = data[target_col].astype(int)

    bool_cols = data.select_dtypes(include="bool").columns
    data[bool_cols] = data[bool_cols].astype(int)

    y = data[target_col]
    X = data.drop(columns=[target_col])

    binary_cols = [
        col for col in X.columns
        if X[col].dropna().nunique() <= 2
    ]
    numerical_cols = [
        col for col in X.columns
        if col not in binary_cols
    ]

    # Fill binary / encoded categorical columns with mode
    for col in binary_cols:
        if X[col].isnull().sum() > 0:
            mode_value = X[col].mode(dropna=True)[0]
            X[col] = X[col].fillna(mode_value)

    # Fill continuous numerical columns with mean
    for col in numerical_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")

        if X[col].isnull().sum() > 0:
            mean_value = X[col].mean()
            X[col] = X[col].fillna(mean_value)

    processed_df = X.copy()
    processed_df[target_col] = y.values

    return processed_df

In [ ]:
df_cleaned = preprocess_missing_values(processed_df)

In [ ]:
df_cleaned.head()

In [ ]:
print(df_cleaned.columns)

In [ ]:
missing_after = df_cleaned.isnull().sum().sort_values(ascending=False)
missing_after.head(20)

In [ ]:
df_cleaned.isnull().sum().sum()

In [ ]:
df_cleaned.shape

In [ ]:
df_cleaned.describe()

# STEP 3: Feature Transformation
we will apply feature transformation because many ICU biomarkers have highly skewed distributions caused by extreme values from critically ill patients. Log transformation helps reduce skewness, stabilize variance, and compress extreme ranges, making the data more suitable for machine learning models while still preserving the clinical severity information.

In [ ]:


  #we will apply log1p transformation to selected skewed features.

def log_transform_features(
    df,
    features_to_transform
):


    data = df.copy()

    for col in features_to_transform:

        if col in data.columns:

            data[col] = np.log1p(data[col])

    return data

now which feature should we transform? well, not all features need transformation. from our KDE graph, we say the features that were heavily skewed, we will be transforming these features.

In [ ]:
skewed_features = [

    "lactate_min",
    "lactate_max",
    "lactate_mean",

    "creatinine_min",
    "creatinine_max",
    "creatinine_mean",

    "bilirubin_min",
    "bilirubin_max",
    "bilirubin_mean",

    "bun_min",
    "bun_max",
    "bun_mean",

    "wbc_min",
    "wbc_max",
    "wbc_mean",

    "urineoutput_24h",

    "glucose_min",
    "glucose_max",
    "glucose_mean"
]

In [ ]:
df_transformed = log_transform_features(
    df_cleaned,
    skewed_features
)

In [ ]:
df_transformed.head()

#STEP 3: TRAIN/TEST SPLIT
Split data: 70% train, 15% validation, 15% test

we will split the dataset into training, validation, and test sets to ensure reliable model development and unbiased evaluation. The training set is used to learn patterns from the data, the validation set is used for hyperparameter tuning and model selection, and the test set is reserved for final performance evaluation on unseen data. Using separate validation and test sets helps prevent overfitting and ensures that the reported model performance reflects true generalization ability.

In [ ]:
df_model = df_transformed.copy()

In [ ]:


target_col = "TARGET_MORTALITY"

X = df_model.drop(columns=[target_col])
y = df_model[target_col].astype(int)

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: split temporary 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)



In [ ]:
print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

In [ ]:
#Lets check class balance in each split
def check_class_distribution(y, name):
    counts = y.value_counts()
    percentages = y.value_counts(normalize=True) * 100

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages.round(2)
    })

    print(f"\n{name} class distribution:")
    print(summary)

check_class_distribution(y_train, "Training")
check_class_distribution(y_val, "Validation")
check_class_distribution(y_test, "Test")

#STEP 4: FEATURE SCALING
Now we are going to scale the features because the clinical variables are measured on very different numerical ranges. For example, pH values are around 7, platelet counts can exceed 1000, and urine output can reach tens of thousands. Without scaling, machine learning models may give too much importance to large-scale variables and ignore smaller-scale but clinically important features. Scaling ensures that all features contribute more fairly to the learning process and improves the performance and stability of many models such as Logistic Regression, SVM, KNN, and PCA.

In [ ]:
#RobustScaler uses the median and interquartile range, so it is less sensitive to extreme clinical values than StandardScaler.

def scale_numerical_features_no_leakage(
    X_train,
    X_val,
    X_test
):

    X_train_scaled = X_train.copy()
    X_val_scaled = X_val.copy()
    X_test_scaled = X_test.copy()

    binary_cols = [
        col for col in X_train.columns
        if X_train[col].nunique() <= 2
    ]

    continuous_cols = [
        col for col in X_train.columns
        if col not in binary_cols
    ]

    scaler = RobustScaler()

    # Fiting ONLY on training data
    X_train_scaled[continuous_cols] = scaler.fit_transform(
        X_train[continuous_cols]
    )

    # Transform validation and test data using training statistics
    X_val_scaled[continuous_cols] = scaler.transform(
        X_val[continuous_cols]
    )

    X_test_scaled[continuous_cols] = scaler.transform(
        X_test[continuous_cols]
    )

    return (
        X_train_scaled,
        X_val_scaled,
        X_test_scaled,
        scaler,
        continuous_cols,
        binary_cols
    )

In [ ]:
X_train_scaled, X_val_scaled, X_test_scaled, scaler, continuous_cols, binary_cols = scale_numerical_features_no_leakage(
    X_train,
    X_val,
    X_test
)

In [ ]:
print("Scaled dataframe shape:", X_train_scaled.shape)
print("Continuous scaled features:", len(continuous_cols))
print("Binary unscaled features:", len(binary_cols))
print("Scaled laidation dataframe shape:", X_val_scaled.shape)
print("Scaled test dataframe shape:", X_test_scaled.shape)

In [ ]:
X_train_scaled.head()

In [ ]:
X_train_scaled[continuous_cols].describe().T.head(20)

In [ ]:


feature = "lactate_max"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before transformation
sns.histplot(
    df_cleaned[feature],
    kde=True,
    ax=axes[0],
    color="crimson"
)

axes[0].set_title(
    f"Before Log Transformation: {feature}",
    fontsize=14,
    fontweight="bold"
)

axes[0].set_xlabel(feature)
axes[0].set_ylabel("Frequency")

# After transformation
sns.histplot(
    X_train_scaled[feature],
    kde=True,
    ax=axes[1],
    color="teal"
)

axes[1].set_title(
    f"After Log Transformation: {feature}",
    fontsize=14,
    fontweight="bold"
)

axes[1].set_xlabel(feature)
axes[1].set_ylabel("Frequency")

plt.suptitle(
    "Effect of Log Transformation on Feature Distribution",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

#STEP 5: Outlier Clipping
We will perform outlier clipping to reduce the influence of extremely large but clinically plausible values that could dominate the learning process and destabilize machine learning models. Instead of removing critically ill patients, clipping caps extreme values within a reasonable range while preserving the underlying clinical information and overall patient records.

in a notshell, we will NOT aggressively remove outliers, Because
the sickest patients are often the most important. If we remove them, we remove mortality signal and the model becomes clinically weaker. so instead of deleting patients, we wil clip extreme values without deleting the patient. to do this, we will use

- IQR-based clipping,

Using:

- Q1−1.5×IQR

and

- Q3+1.5×IQR

In [ ]:

def clip_outliers_iqr_no_leakage(
    X_train,
    X_val,
    X_test
):

    X_train_clipped = X_train.copy()
    X_val_clipped = X_val.copy()
    X_test_clipped = X_test.copy()

    continuous_cols = [
        col for col in X_train.columns
        if X_train[col].nunique() > 2
    ]

    clipping_summary = []

    for col in continuous_cols:

        Q1 = X_train[col].quantile(0.25)
        Q3 = X_train[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Count clipped values in each set
        train_outliers = (
            (X_train[col] < lower_bound) |
            (X_train[col] > upper_bound)
        ).sum()

        val_outliers = (
            (X_val[col] < lower_bound) |
            (X_val[col] > upper_bound)
        ).sum()

        test_outliers = (
            (X_test[col] < lower_bound) |
            (X_test[col] > upper_bound)
        ).sum()

        # Apply clipping using TRAIN bounds
        X_train_clipped[col] = X_train[col].clip(
            lower=lower_bound,
            upper=upper_bound
        )

        X_val_clipped[col] = X_val[col].clip(
            lower=lower_bound,
            upper=upper_bound
        )

        X_test_clipped[col] = X_test[col].clip(
            lower=lower_bound,
            upper=upper_bound
        )

        clipping_summary.append({
            "Feature": col,
            "Lower_Bound": lower_bound,
            "Upper_Bound": upper_bound,
            "Train_Num_Clipped": train_outliers,
            "Train_Percent_Clipped": 100 * train_outliers / len(X_train),
            "Val_Num_Clipped": val_outliers,
            "Val_Percent_Clipped": 100 * val_outliers / len(X_val),
            "Test_Num_Clipped": test_outliers,
            "Test_Percent_Clipped": 100 * test_outliers / len(X_test)
        })

    clipping_summary_df = pd.DataFrame(clipping_summary).sort_values(
        by="Train_Percent_Clipped",
        ascending=False
    )

    return (
        X_train_clipped,
        X_val_clipped,
        X_test_clipped,
        clipping_summary_df
    )

In [ ]:
X_train_clipped, X_val_clipped, X_test_clipped, clipping_summary = clip_outliers_iqr_no_leakage(
    X_train,
    X_val,
    X_test
)

In [ ]:
clipping_summary.head(20)

In [ ]:


top_clip = clipping_summary.head(20)

plt.figure(figsize=(14,8))

ax = sns.barplot(
    data=top_clip,
    y="Feature",
    x="Train_Percent_Clipped",
    palette="magma"
)

plt.title(
    "Top Features Requiring Outlier Clipping",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel("Percentage of Clipped Values (%)", fontsize=14)
plt.ylabel("Feature", fontsize=14)

for p in ax.patches:

    width = p.get_width()

    ax.annotate(
        f"{width:.1f}%",
        (width, p.get_y() + p.get_height()/2),
        ha="left",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

sns.despine()

plt.tight_layout()

plt.show()

In [ ]:
!pip install -q imbalanced-learn

#step 7: Class Imbalance Handling (SMOTE Strategy)

Now, we have to smote because the mortality dataset is imbalanced, with survivor cases being much more frequent than death cases. Without balancing, machine learning models may become biased toward the majority class and fail to correctly identify high-risk septic patients. SMOTE helps address this problem by generating synthetic minority-class samples, allowing the model to learn mortality-related patterns more effectively and improving metrics such as sensitivity, recall, and F1-score.

In [ ]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_clipped,
    y_train
)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

In [ ]:
#lets visualise this
def plot_class_balance(y_before, y_after):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.countplot(x=y_before, ax=axes[0], palette=["#2E7D32", "#C62828"])
    axes[0].set_title("Training Class Distribution Before SMOTE", fontweight="bold")
    axes[0].set_xlabel("Mortality Class")
    axes[0].set_ylabel("Count")
    axes[0].set_xticklabels(["Survived", "Died"])

    sns.countplot(x=y_after, ax=axes[1], palette=["#2E7D32", "#C62828"])
    axes[1].set_title("Training Class Distribution After SMOTE", fontweight="bold")
    axes[1].set_xlabel("Mortality Class")
    axes[1].set_ylabel("Count")
    axes[1].set_xticklabels(["Survived", "Died"])

    plt.tight_layout()
    plt.show()

plot_class_balance(y_train, y_train_smote)

#STEP 8: LEAKAGE PREVENTION

Data leakage occurs when information from the validation or test datasets unintentionally influences the training process, leading to overly optimistic model performance and poor generalization to unseen patients. In healthcare machine learning, leakage is particularly dangerous because it can produce models that appear highly accurate during development but fail in real clinical settings.

To prevent leakage, the dataset was first split into training, validation, and test sets before applying any final preprocessing or resampling techniques. All preprocessing operations, including scaling, and outlier clipping, were fitted only on the training set and then applied to the validation and test sets using the same learned parameters. Additionally, SMOTE was applied exclusively to the training set to avoid introducing synthetic information into the validation and test data. This ensured that model evaluation remained unbiased and reflected true predictive generalization.

Feature engineering was applied by transforming raw time-series ICU data into fixed patient-level features. For each vital sign and laboratory biomarker, minimum, maximum, and mean values over the first 24 hours were extracted to capture both typical physiological state and extreme deterioration. Additional clinically meaningful features such as SOFA score, vasopressor use, and 24-hour urine output were also included to represent organ dysfunction and treatment intensity

We deliberately restricted feature extraction to the first 24 hours of ICU admission because the primary objective of the project was early mortality prediction in septic patients. Clinically, the first 24 hours are critical for identifying hemodynamic instability, organ dysfunction, and early treatment response. Using only early ICU data makes the model more clinically actionable, since clinicians need risk estimates as early as possible to guide interventions and resource allocation.


Although limiting the analysis to the first 24 hours may lead to some loss of information regarding later physiological deterioration, extending the observation window to the entire ICU stay may introduce temporal leakage, where the model indirectly learns from events occurring close to death. By focusing on the first 24 hours, we aimed to balance predictive performance with clinical realism and fair model evaluation.

To reduce information loss within the 24-hour window, we engineered multiple aggregation features including minimum, maximum, and mean values for each physiological variable. This allowed the model to capture baseline state, overall trend, and extreme deterioration during the early critical phase of sepsis.

#PART 3: FEATURE SELECTION AND MODEL DEVELOPMENT

now, we have to reduce redundancy by applying feature selection to improve model interpretability, and minimize overfitting by removing highly correlated or less informative variables. In clinical machine learning, it is important to preserve meaningful biomarkers while simplifying the feature space so that the model can generalize better and remain clinically interpretable.

#PART 3A: FEATURE SELECTION

In [ ]:


# Correlation matrix
corr_matrix = X_train_clipped.corr().abs()

# Upper triangle
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Features above threshold
threshold = 0.90

high_corr_features = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > threshold)
]

print("Highly correlated features:")
print(high_corr_features)

print("\nNumber of highly correlated features:")
print(len(high_corr_features))

In [ ]:


strong_corr = (
    upper_triangle.stack()
    .reset_index()
)

strong_corr.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

strong_corr = strong_corr[
    strong_corr["Correlation"] > threshold
]

strong_corr = strong_corr.sort_values(
    by="Correlation",
    ascending=False
)

plt.figure(figsize=(12,8))

ax = sns.barplot(
    data=strong_corr.head(20),
    y="Feature_1",
    x="Correlation",
    palette="viridis"
)

plt.title(
    "Top Highly Correlated Features",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Absolute Correlation", fontsize=13)
plt.ylabel("Feature", fontsize=13)

sns.despine()

plt.tight_layout()

plt.show()

In [ ]:
features_to_remove = [

    "lactate_mean",

    "wbc_max",

    "creatinine_mean",

    "bilirubin_mean",

    "platelet_max",
    "platelet_mean",

    "bun_mean",

    "sodium_mean",

    "bicarbonate_mean",

    "glucose_mean",

    "hemoglobin_mean"
]

In [ ]:
X_train_selected = X_train_clipped.drop(
    columns=features_to_remove,
    errors="ignore"
)

X_val_selected = X_val_clipped.drop(
    columns=features_to_remove,
    errors="ignore"
)

X_test_selected = X_test_clipped.drop(
    columns=features_to_remove,
    errors="ignore"
)

X_train_smote_selected = X_train_smote.drop(
    columns=features_to_remove,
    errors="ignore"
)

print("Remaining features:",
      X_train_selected.shape[1])

In [ ]:
print("Remaining features:",
      X_train_smote_selected.shape[1])

Correlation analysis revealed several highly redundant engineered biomarkers, particularly among the minimum, maximum, and mean aggregations derived from the same physiological variable. A correlation threshold of 0.90 was used to identify strongly redundant features because correlations above this level generally indicate that the variables contain nearly identical information. Using this threshold helped reduce redundancy and model complexity while preserving clinically meaningful variability and interpretability.

Feature removal was not performed blindly based only on statistics. Instead, both clinical relevance and physiological interpretation were considered. Features such as lactate_mean, creatinine_mean, bilirubin_mean, platelet_mean, and bun_mean were removed because they were highly correlated with more clinically informative severity indicators such as maximum or minimum values. In sepsis, extreme physiological deterioration often carries stronger prognostic value than average measurements. For example, peak lactate reflects severe tissue hypoperfusion, maximum creatinine reflects acute kidney injury severity, and minimum platelet count reflects septic coagulopathy and disseminated intravascular coagulation (DIC).

Despite strong correlations, some related features were intentionally retained because they capture complementary clinical information. For example, both lactate_max and lactate_min may represent different aspects of perfusion dynamics, while minimum blood pressure measurements are clinically important for identifying septic shock and hemodynamic instability. Therefore, feature selection combined statistical redundancy analysis with clinical reasoning to preserve biomarkers that are physiologically meaningful and predictive of organ dysfunction and mortality in sepsis patients.

#PART 3B: MODEL DEVELOPMENT

Now is the time to build the model, first, lets check correctness

In [ ]:
X_train_selected.isnull().sum().sum()
X_val_selected.isnull().sum().sum()
X_test_selected.isnull().sum().sum()
X_train_smote_selected.isnull().sum().sum()

ok, all is correct

#We will be using 3 models, LOGISTIC REGRESSION, SVM AND NEURAL NETWORK.

#MODEL 1: LOGISTIC REGRESSION

FIRST, WE FIT

In [ ]:
# Logistic Regression baseline
log_reg_baseline = LogisticRegression(
    max_iter=5000,
    random_state=42,
    class_weight=None
)

In [ ]:
log_reg_baseline.fit(X_train_smote_selected, y_train_smote)

In [ ]:
y_val_pred = log_reg_baseline.predict(X_val_selected)
y_val_proba = log_reg_baseline.predict_proba(X_val_selected)[:, 1]

SECOND, WE EVALUATE

In [ ]:

def evaluate_classifier(y_true, y_pred, y_proba, model_name="Model"):
    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()

    specificity = tn / (tn + fp)
    sensitivity = recall_score(y_true, y_pred)

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall/Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1-score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "PR-AUC": average_precision_score(y_true, y_proba)
    }

    print(f"\n===== {model_name} =====")
    print(pd.DataFrame([results]))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(cm)

    return results

In [ ]:
log_reg_baseline_results = evaluate_classifier(
    y_val,
    y_val_pred,
    y_val_proba,
    model_name="Logistic Regression Baseline"
)

In [ ]:

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8,6)




#confusion matrix

In [ ]:

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Predicted Survived", "Predicted Died"],
        yticklabels=["Actual Survived", "Actual Died"]
    )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_confusion_matrix(
    y_val,
    y_val_pred,
    title="Logistic Regression Baseline - Validation Confusion Matrix"
)

In [ ]:

# ROC CURVE

fpr, tpr, _ = roc_curve(y_val, y_val_proba)
auc_score = roc_auc_score(y_val, y_val_proba)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    linewidth=3,
    label=f"ROC-AUC = {auc_score:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    linewidth=2
)

plt.title(
    "Logistic Regression — ROC Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "False Positive Rate",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Positive Rate (Sensitivity)",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# PRECISION-RECALL CURVE


precision, recall, _ = precision_recall_curve(
    y_val,
    y_val_proba
)

pr_auc = average_precision_score(
    y_val,
    y_val_proba
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    linewidth=3,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.title(
    "Logistic Regression — Precision-Recall Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Recall",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "Precision",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# METRICS BARPLOT

metrics_dict = {

    "Accuracy": 0.726505,
    "Precision": 0.559871,
    "Recall": 0.584459,
    "Specificity": 0.791091,
    "F1-score": 0.571901,
    "ROC-AUC": 0.761116,
    "PR-AUC": 0.627818
}

metrics_df = pd.DataFrame({

    "Metric": list(metrics_dict.keys()),
    "Score": list(metrics_dict.values())

})

plt.figure(figsize=(10,6))

ax = sns.barplot(
    data=metrics_df,
    x="Metric",
    y="Score",
    palette="viridis"
)

plt.ylim(0,1)

plt.title(
    "Logistic Regression — Performance Metrics",
    fontsize=20,
    fontweight="bold"
)

plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel(
    "",
    fontsize=14
)

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.xticks(rotation=20, fontsize=11)

sns.despine()

plt.tight_layout()
plt.show()

#Insight

The Logistic Regression baseline model demonstrated moderate predictive performance for sepsis mortality prediction. The model achieved an accuracy of approximately 73%, but more importantly, it obtained a ROC-AUC of 0.76, indicating a reasonably good ability to distinguish between survivors and non-survivors across different classification thresholds. Since the dataset is clinically imbalanced, ROC-AUC, recall, and PR-AUC are more informative than accuracy alone.

From the confusion matrix, the model correctly identified 173 mortality cases (true positives) while missing 123 mortality cases (false negatives). In a clinical setting, false negatives are particularly important because they correspond to septic patients predicted as low-risk despite eventual mortality. Nevertheless, the recall (sensitivity) of approximately 58% shows that the model is capable of identifying more than half of the high-risk patients, which is acceptable for an initial baseline model.

The specificity of 79% indicates that the model performs better at identifying survivors than non-survivors. This behavior is expected in medical mortality prediction tasks where the non-mortality class is more represented. The precision-recall curve also demonstrates stable predictive behavior, with a PR-AUC of approximately 0.63, suggesting that the model retains reasonable precision while identifying mortality cases.

Clinically, the model performance suggests that the selected biomarkers and preprocessing pipeline successfully captured meaningful patterns associated with severe sepsis, organ dysfunction, and mortality risk. However, the moderate sensitivity indicates that more advanced nonlinear models such as SVMs and neural networks may further improve detection of complex physiological relationships and critically ill patients.

#Logistic Regression Hyperparameter Tuning

we will tune the C parameter because C controls regularization strength.
- small C → stronger regularization
- large C → weaker regularization

In [ ]:

#parameter grid
param_grid_logreg_fast = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["liblinear"],
    "class_weight": [None]
}

log_reg = LogisticRegression(
    max_iter=2000,
    random_state=42
)

grid_logreg_fast = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_logreg_fast,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_logreg_fast.fit(
    X_train_smote_selected,
    y_train_smote
)

BEST HYPERPARAMETERS

In [ ]:
print("Best Parameters:", grid_logreg_fast.best_params_)
print("Best CV ROC-AUC:", grid_logreg_fast.best_score_)

#Lets get the best model

In [ ]:
best_logreg = grid_logreg_fast.best_estimator_

y_val_pred_tuned = best_logreg.predict(X_val_selected)
y_val_proba_tuned = best_logreg.predict_proba(X_val_selected)[:, 1]

tuned_logreg_results = evaluate_classifier(
    y_val,
    y_val_pred_tuned,
    y_val_proba_tuned,
    model_name="Tuned Logistic Regression"
)

#Visualisation after hyperparameter tune

In [ ]:


sns.set_style("whitegrid")


# CONFUSION MATRIX

cm = confusion_matrix(y_val, y_val_pred_tuned)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=2,
    linecolor="white",
    cbar=True,
    annot_kws={"fontsize":18, "fontweight":"bold"}
)

plt.title(
    "Tuned Logistic Regression — Confusion Matrix",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Label",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Label",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    [0.5, 1.5],
    ["Predicted Survived", "Predicted Died"],
    fontsize=12
)

plt.yticks(
    [0.5, 1.5],
    ["Actual Survived", "Actual Died"],
    fontsize=12,
    rotation=0
)

plt.tight_layout()
plt.show()




In [ ]:

# ROC CURVE


fpr, tpr, _ = roc_curve(
    y_val,
    y_val_proba_tuned
)

auc_score = roc_auc_score(
    y_val,
    y_val_proba_tuned
)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    linewidth=3,
    label=f"ROC-AUC = {auc_score:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    linewidth=2
)

plt.title(
    "Tuned Logistic Regression — ROC Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "False Positive Rate",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Positive Rate (Sensitivity)",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:


# PRECISION-RECALL CURVE

precision, recall, _ = precision_recall_curve(
    y_val,
    y_val_proba_tuned
)

pr_auc = average_precision_score(
    y_val,
    y_val_proba_tuned
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    linewidth=3,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.title(
    "Tuned Logistic Regression — Precision-Recall Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Recall",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "Precision",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# PERFORMANCE METRICS BARPLOT

metrics_dict = {

    "Accuracy": 0.740232,
    "Precision": 0.583333,
    "Recall": 0.591216,
    "Specificity": 0.807988,
    "F1-score": 0.587248,
    "ROC-AUC": 0.768599,
    "PR-AUC": 0.627698
}

metrics_df = pd.DataFrame({

    "Metric": list(metrics_dict.keys()),
    "Score": list(metrics_dict.values())

})

plt.figure(figsize=(10,6))

ax = sns.barplot(
    data=metrics_df,
    x="Metric",
    y="Score",
    palette="viridis"
)

plt.ylim(0,1)

plt.title(
    "Tuned Logistic Regression — Performance Metrics",
    fontsize=20,
    fontweight="bold"
)

plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("")

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.xticks(rotation=20, fontsize=11)

sns.despine()

plt.tight_layout()
plt.show()

#Comparing Baseline vs Tuned Model

In [ ]:
comparison_df = pd.DataFrame([

    log_reg_baseline_results,

    tuned_logreg_results

])

comparison_df

In [ ]:
comparison_melted = comparison_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12,6))

sns.barplot(
    data=comparison_melted,
    x="Metric",
    y="Score",
    hue="Model"
)

plt.ylim(0,1)

plt.title(
    "Baseline vs Tuned Logistic Regression",
    fontsize=18,
    fontweight="bold"
)

plt.xticks(rotation=20)

plt.tight_layout()

plt.show()

FEATURE IMPORTANCE FROM LOGISTIC REGRESSION

Logistic Regression is GOOD because coefficients are interpretable.we will be able to identify:

- which biomarkers increase mortality risk,
- which reduce risk.

This is VERY valuable clinically.

In [ ]:
coef_df = pd.DataFrame({

    "Feature": X_train_smote_selected.columns,

    "Coefficient": best_logreg.coef_[0]

})

coef_df["Abs_Coefficient"] = np.abs(
    coef_df["Coefficient"]
)

coef_df = coef_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

coef_df.head(20)

In [ ]:
top_coef = coef_df.head(20)

plt.figure(figsize=(10,8))

sns.barplot(
    data=top_coef,
    y="Feature",
    x="Coefficient",
    palette="coolwarm"
)

plt.title(
    "Top Logistic Regression Features",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.show()

After hyperparameter tuning, the Logistic Regression model showed a modest but consistent improvement in performance compared to the baseline model. The ROC-AUC increased from approximately 0.761 to 0.769, while accuracy improved from 72.7% to 74.0%. Sensitivity (recall) also slightly increased, indicating a better ability to identify high-risk septic patients. In addition, specificity improved to approximately 81%, showing stronger discrimination of survivor cases.

The optimal hyperparameters selected by GridSearchCV were:

C = 10
penalty = L2
solver = liblinear

This suggests that a relatively low level of regularization allowed the model to better capture clinically relevant relationships within the feature space without severe overfitting. The improvement after tuning demonstrates the importance of systematic hyperparameter optimization rather than relying on default parameters.

Clinically, the tuned model exhibited a more balanced trade-off between detecting mortality cases and minimizing false alarms. However, the moderate sensitivity indicates that mortality prediction in sepsis remains a complex nonlinear problem, motivating the exploration of more advanced models such as SVMs and neural networks.

#MODEL 2: Support Vector Machine (SVM)

BASELINE SVM

In [ ]:

svm_baseline = SVC(

    kernel="rbf",

    probability=True,

    random_state=42
)

# Train
svm_baseline.fit(

    X_train_smote_selected,

    y_train_smote
)



In [ ]:

y_val_pred_svm = svm_baseline.predict(
    X_val_selected
)

y_val_proba_svm = svm_baseline.predict_proba(
    X_val_selected
)[:,1]

EVALUATE BASELINE SVM

In [ ]:
svm_baseline_results = evaluate_classifier(

    y_val,

    y_val_pred_svm,

    y_val_proba_svm,

    model_name="Baseline SVM"
)

In [ ]:

sns.set_style("whitegrid")


# CONFUSION MATRIX


cm = confusion_matrix(y_val, y_val_pred_svm)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Purples",
    linewidths=2,
    linecolor="white",
    cbar=True,
    annot_kws={"fontsize":18, "fontweight":"bold"}
)

plt.title(
    "Baseline SVM — Confusion Matrix",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Label",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Label",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    [0.5, 1.5],
    ["Predicted Survived", "Predicted Died"],
    fontsize=12
)

plt.yticks(
    [0.5, 1.5],
    ["Actual Survived", "Actual Died"],
    fontsize=12,
    rotation=0
)

plt.tight_layout()
plt.show()

In [ ]:

# ROC CURVE


fpr, tpr, _ = roc_curve(
    y_val,
    y_val_proba_svm
)

auc_score = roc_auc_score(
    y_val,
    y_val_proba_svm
)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    linewidth=3,
    label=f"ROC-AUC = {auc_score:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    linewidth=2
)

plt.title(
    "Baseline SVM — ROC Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "False Positive Rate",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Positive Rate (Sensitivity)",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# PRECISION-RECALL CURVE

precision, recall, _ = precision_recall_curve(
    y_val,
    y_val_proba_svm
)

pr_auc = average_precision_score(
    y_val,
    y_val_proba_svm
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    linewidth=3,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.title(
    "Baseline SVM — Precision-Recall Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Recall",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "Precision",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# PERFORMANCE METRICS BARPLOT

metrics_dict = {

    "Accuracy": 0.644139,
    "Precision": 0.447570,
    "Recall": 0.591216,
    "Specificity": 0.668203,
    "F1-score": 0.509461,
    "ROC-AUC": 0.692544,
    "PR-AUC": 0.543834
}

metrics_df = pd.DataFrame({

    "Metric": list(metrics_dict.keys()),
    "Score": list(metrics_dict.values())

})

plt.figure(figsize=(10,6))

ax = sns.barplot(
    data=metrics_df,
    x="Metric",
    y="Score",
    palette="magma"
)

plt.ylim(0,1)

plt.title(
    "Baseline SVM — Performance Metrics",
    fontsize=20,
    fontweight="bold"
)

plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("")

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.xticks(rotation=20, fontsize=11)

sns.despine()

plt.tight_layout()
plt.show()

The baseline SVM model achieved moderate predictive performance but performed worse than the Logistic Regression baseline across most evaluation metrics. The model obtained a ROC-AUC of approximately 0.69, indicating weaker discrimination between survivors and non-survivors compared to Logistic Regression. Although the sensitivity remained relatively similar (≈59%), the precision and specificity were lower, suggesting that the model generated more false positive mortality predictions.

The confusion matrix shows that the SVM correctly identified 175 mortality cases but incorrectly classified 216 survivors as high-risk deaths. This indicates that the default SVM hyperparameters produced a decision boundary that was not yet optimally adapted to the clinical feature space.

Clinically, the model still captured important mortality-related physiological patterns, but the lower ROC-AUC and PR-AUC suggest that the nonlinear relationships within the dataset were not fully exploited using the default configuration. This motivates systematic hyperparameter tuning of the SVM model, particularly the C and gamma parameters, which strongly influence the flexibility and complexity of the decision boundary.

#SVM HYPERPARAMTERS TUNING
we will tune C and gamma values

In [ ]:
param_grid_svm = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.01, 0.1],
    "kernel": ["rbf"]
}

svm_model = SVC(
    probability=True,
    random_state=42
)

grid_svm = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid_svm,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_svm.fit(
    X_train_smote_selected,
    y_train_smote
)



In [ ]:
print("Best Parameters:", grid_svm.best_params_)
print("Best CV ROC-AUC:", grid_svm.best_score_)

#evaluation

In [ ]:
best_svm = grid_svm.best_estimator_

y_val_pred_svm_tuned = best_svm.predict(X_val_selected)
y_val_proba_svm_tuned = best_svm.predict_proba(X_val_selected)[:, 1]

svm_tuned_results = evaluate_classifier(
    y_val,
    y_val_pred_svm_tuned,
    y_val_proba_svm_tuned,
    model_name="Tuned SVM"
)

In [ ]:
sns.set_style("whitegrid")

# =========================================================
# CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(y_val, y_val_pred_svm_tuned)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Reds",
    linewidths=2,
    linecolor="white",
    cbar=True,
    annot_kws={"fontsize":18, "fontweight":"bold"}
)

plt.title(
    "Tuned SVM — Confusion Matrix",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Label",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Label",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    [0.5, 1.5],
    ["Predicted Survived", "Predicted Died"],
    fontsize=12
)

plt.yticks(
    [0.5, 1.5],
    ["Actual Survived", "Actual Died"],
    fontsize=12,
    rotation=0
)

plt.tight_layout()
plt.show()

In [ ]:

# ROC CURVE

fpr, tpr, _ = roc_curve(
    y_val,
    y_val_proba_svm_tuned
)

auc_score = roc_auc_score(
    y_val,
    y_val_proba_svm_tuned
)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    linewidth=3,
    label=f"ROC-AUC = {auc_score:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    linewidth=2
)

plt.title(
    "Tuned SVM — ROC Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "False Positive Rate",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Positive Rate (Sensitivity)",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# PRECISION-RECALL CURVE

precision, recall, _ = precision_recall_curve(
    y_val,
    y_val_proba_svm_tuned
)

pr_auc = average_precision_score(
    y_val,
    y_val_proba_svm_tuned
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    linewidth=3,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.title(
    "Tuned SVM — Precision-Recall Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Recall",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "Precision",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# PERFORMANCE METRICS BARPLOT

metrics_dict = {

    "Accuracy": 0.687434,
    "Precision": 0.500000,
    "Recall": 0.003378,
    "Specificity": 0.998464,
    "F1-score": 0.006711,
    "ROC-AUC": 0.574592,
    "PR-AUC": 0.340573
}

metrics_df = pd.DataFrame({

    "Metric": list(metrics_dict.keys()),
    "Score": list(metrics_dict.values())

})

plt.figure(figsize=(10,6))

ax = sns.barplot(
    data=metrics_df,
    x="Metric",
    y="Score",
    palette="rocket"
)

plt.ylim(0,1)

plt.title(
    "Tuned SVM — Performance Metrics",
    fontsize=20,
    fontweight="bold"
)

plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("")

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.xticks(rotation=20, fontsize=11)

sns.despine()

plt.tight_layout()
plt.show()

Despite achieving a very high cross-validation ROC-AUC during tuning (≈0.93), the tuned SVM generalized very poorly on the validation dataset. The model almost completely failed to identify mortality cases, with a recall of only 0.3%, meaning that nearly all non-survivors were misclassified as survivors. This is clearly visible in the confusion matrix, where only 1 mortality case was correctly detected out of 296.

The extremely high specificity (≈99.8%) indicates that the model became heavily biased toward predicting the survivor class. This behavior suggests severe overfitting to the SMOTE training distribution, where the selected hyperparameters (C=10, gamma=0.01) likely produced an overly rigid or poorly calibrated decision boundary.

Clinically, this model would be unsafe because missing critically ill septic patients is far more dangerous than generating additional false alarms. Although the training cross-validation score appeared excellent, the validation performance demonstrates the importance of evaluating models on an independent validation set rather than relying solely on cross-validation results. This also highlights a key lesson in medical machine learning: strong cross-validation performance does not necessarily imply good clinical generalization.

#lets compare baseine with thw hyperparameter tune

In [ ]:

# BASELINE vs TUNED SVM COMPARISON

svm_comparison_df = pd.DataFrame([

    svm_baseline_results,

    svm_tuned_results

])

svm_comparison_df

In [ ]:


svm_comparison_melted = svm_comparison_df.melt(

    id_vars="Model",

    var_name="Metric",

    value_name="Score"
)

plt.figure(figsize=(14,6))

ax = sns.barplot(

    data=svm_comparison_melted,

    x="Metric",

    y="Score",

    hue="Model",

    palette="magma"
)

plt.ylim(0,1)

plt.title(
    "Baseline vs Tuned SVM Performance",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "",
    fontsize=14
)

plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    rotation=20,
    fontsize=11
)

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

plt.legend(
    title="SVM Model",
    fontsize=11
)

sns.despine()

plt.tight_layout()

plt.show()

The comparison between the baseline and tuned SVM models shows that hyperparameter tuning did not improve generalization performance on the validation set. Although the tuned SVM achieved a very high cross-validation ROC-AUC during training, its validation sensitivity collapsed, indicating severe overfitting and poor clinical generalization. The tuned model became heavily biased toward predicting survivor cases, leading to extremely high specificity but near-zero recall for mortality detection.

In contrast, the baseline SVM maintained a more balanced trade-off between sensitivity and specificity, even though its overall performance remained lower than Logistic Regression. These findings highlight the importance of independent validation in biomedical machine learning and demonstrate that increased model complexity does not necessarily translate to better clinical prediction performance.

#MODEL 3: NEURAL NETWORK WITH PYTORCH

STEP 1 : PREPARING PYTORCH TENSORS

In [ ]:
import torch

# Convert to tensors
X_train_tensor = torch.tensor(
    X_train_smote_selected.values,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_smote.values,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_selected.values,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
)

STEP 2 : CREATING DATALOADERS

In [ ]:


train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

STEP 3 : BUILDING NEURAL NETWORK

In [ ]:

class SepsisNN(nn.Module):

    def __init__(self, input_dim):

        super(SepsisNN, self).__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64, 32),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )


    def forward(self, x):

        return self.network(x)

LETS INITIALISE THE MODEL

In [ ]:
input_dim = X_train_smote_selected.shape[1]

model = SepsisNN(input_dim)

model

STEP 4 — LOSS FUNCTION + OPTIMIZER

In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

STEP 5:  TRAINING LOOP

In [ ]:
num_epochs = 50

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    # TRAINING

    model.train()

    running_train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze()

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = (
        running_train_loss / len(train_loader)
    )

    train_losses.append(avg_train_loss)

    # VALIDATION

    model.eval()

    running_val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch).squeeze()

            loss = criterion(
                outputs,
                y_batch
            )

            running_val_loss += loss.item()

    avg_val_loss = (
        running_val_loss / len(val_loader)
    )

    val_losses.append(avg_val_loss)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {avg_train_loss:.4f} "
        f"Val Loss: {avg_val_loss:.4f}"
    )

STEP 6 : LOSS CURVES

- This helps detect:

overfitting,
underfitting.

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    label="Training Loss",
    linewidth=2
)

plt.plot(
    val_losses,
    label="Validation Loss",
    linewidth=2
)

plt.title(
    "Neural Network Training Curve",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

The neural network appears capable of learning clinically meaningful mortality patterns while maintaining relatively stable generalization performance on unseen validation patients. The absence of severe divergence between training and validation losses suggests that the preprocessing pipeline, feature selection strategy, and dropout regularization were effective in reducing overfitting.

MODEL EVALUATION

In [ ]:

model.eval()

with torch.no_grad():
    val_logits = model(X_val_tensor).squeeze()
    val_probs = torch.sigmoid(val_logits).cpu().numpy()

# Default threshold = 0.5
y_val_pred_nn = (val_probs >= 0.5).astype(int)
y_val_proba_nn = val_probs

In [ ]:
nn_baseline_results = evaluate_classifier(
    y_val,
    y_val_pred_nn,
    y_val_proba_nn,
    model_name="Baseline Neural Network"
)

In [ ]:
plot_confusion_matrix(
    y_val,
    y_val_pred_nn,
    title="Baseline Neural Network - Validation Confusion Matrix"
)

In [ ]:
plot_roc_curve(
    y_val,
    y_val_proba_nn,
    title="Baseline Neural Network - ROC Curve"
)

In [ ]:


precision, recall, _ = precision_recall_curve(y_val, y_val_proba_nn)
pr_auc = average_precision_score(y_val, y_val_proba_nn)

plt.figure(figsize=(8,6))
plt.plot(recall, precision, linewidth=3, label=f"PR-AUC = {pr_auc:.3f}")
plt.title("Baseline Neural Network - Precision-Recall Curve", fontsize=18, fontweight="bold")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
nn_metrics_df = pd.DataFrame([nn_baseline_results]).drop(columns=["Model"])

nn_metrics_melted = nn_metrics_df.melt(
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10,6))

ax = sns.barplot(
    data=nn_metrics_melted,
    x="Metric",
    y="Score",
    palette="crest"
)

plt.ylim(0,1)

plt.title(
    "Baseline Neural Network — Performance Metrics",
    fontsize=18,
    fontweight="bold"
)

plt.ylabel("Score", fontsize=13)
plt.xlabel("")

for p in ax.patches:
    height = p.get_height()
    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

plt.xticks(rotation=20)
sns.despine()
plt.tight_layout()
plt.show()

The baseline Neural Network achieved solid and stable performance on the validation dataset, with a ROC-AUC of approximately 0.76 and an accuracy of 70%. Compared to the SVM model, the neural network generalized substantially better and maintained a balanced trade-off between sensitivity and specificity. The model correctly identified 168 mortality cases while maintaining acceptable survivor classification performance.

Clinically, the neural network successfully captured nonlinear relationships among physiological biomarkers associated with sepsis mortality, including hemodynamic instability, organ dysfunction, and metabolic abnormalities. However, despite its nonlinear learning capability, the model did not significantly outperform the tuned Logistic Regression model, whose ROC-AUC remained slightly higher.

This observation is scientifically important because it suggests that the mortality patterns in the dataset may already be reasonably represented through linear relationships among strong clinical biomarkers such as lactate, SOFA score, blood pressure, renal function markers, and vasopressor usage. Nevertheless, the neural network demonstrated more stable nonlinear learning behavior than SVM and remains a strong candidate for further architectural tuning and optimization.

#TUNING THE HYPERPARAMTERS OF THE NEURAL NETWORK

In [ ]:
# Flexible NN architecture

class TunableSepsisNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout):
        super(TunableSepsisNN, self).__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:

# Training function with early stopping

def train_nn_model(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_layers,
    dropout,
    learning_rate,
    batch_size,
    max_epochs=100,
    patience=10
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

    X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    input_dim = X_train.shape[1]

    model = TunableSepsisNN(
        input_dim=input_dim,
        hidden_layers=hidden_layers,
        dropout=dropout
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    best_val_loss = np.inf
    best_model_state = None
    patience_counter = 0

    train_losses = []
    val_losses = []

    for epoch in range(max_epochs):

        model.train()
        running_train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation loss
        model.eval()

        with torch.no_grad():
            X_val_device = X_val_tensor.to(device)
            y_val_device = y_val_tensor.to(device)

            val_logits = model(X_val_device).squeeze()
            val_loss = criterion(val_logits, y_val_device).item()

        val_losses.append(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    # Load best model
    model.load_state_dict(best_model_state)


    model.eval()

    with torch.no_grad():
        val_logits = model(X_val_tensor.to(device)).squeeze()
        val_probs = torch.sigmoid(val_logits).cpu().numpy()

    val_preds = (val_probs >= 0.5).astype(int)

    results = {
        "Hidden_Layers": str(hidden_layers),
        "Dropout": dropout,
        "Learning_Rate": learning_rate,
        "Batch_Size": batch_size,
        "Best_Val_Loss": best_val_loss,
        "Epochs_Trained": epoch + 1,
        "Accuracy": accuracy_score(y_val, val_preds),
        "Precision": precision_score(y_val, val_preds, zero_division=0),
        "Recall": recall_score(y_val, val_preds, zero_division=0),
        "F1": f1_score(y_val, val_preds, zero_division=0),
        "ROC_AUC": roc_auc_score(y_val, val_probs),
        "PR_AUC": average_precision_score(y_val, val_probs)
    }

    return model, results, train_losses, val_losses

In [ ]:

# Hyperparameter grid

hidden_layer_options = [
    [64, 32],
    [128, 64],
    [128, 64, 32]
]

dropout_options = [0.2, 0.3, 0.5]
learning_rate_options = [0.001, 0.0005]
batch_size_options = [32, 64]

experiments = list(itertools.product(
    hidden_layer_options,
    dropout_options,
    learning_rate_options,
    batch_size_options
))

print("Total experiments:", len(experiments))

In [ ]:
# Run tuning

all_results = []
best_model = None
best_score = -np.inf
best_losses = None

for i, (hidden_layers, dropout, lr, batch_size) in enumerate(experiments):

    print(f"\nExperiment {i+1}/{len(experiments)}")
    print("Hidden:", hidden_layers, "| Dropout:", dropout, "| LR:", lr, "| Batch:", batch_size)

    model, results, train_losses, val_losses = train_nn_model(
        X_train_smote_selected,
        y_train_smote,
        X_val_selected,
        y_val,
        hidden_layers=hidden_layers,
        dropout=dropout,
        learning_rate=lr,
        batch_size=batch_size,
        max_epochs=100,
        patience=10
    )

    all_results.append(results)

    # Selection score: prioritize ROC-AUC, then PR-AUC and Recall
    selection_score = (
        0.5 * results["ROC_AUC"] +
        0.3 * results["PR_AUC"] +
        0.2 * results["Recall"]
    )

    if selection_score > best_score:
        best_score = selection_score
        best_model = model
        best_losses = (train_losses, val_losses)
        best_config = results

In [ ]:

# Results table

nn_tuning_results = pd.DataFrame(all_results)

nn_tuning_results = nn_tuning_results.sort_values(
    by=["ROC_AUC", "PR_AUC", "Recall"],
    ascending=False
)

nn_tuning_results.head(10)

check the best configuration

In [ ]:
best_config

lets save the best model

In [ ]:
torch.save(best_model.state_dict(), "best_neural_network_model.pth")

#ROC-AUC Progression Across Experiments

In [ ]:


sns.set_style("whitegrid")

plot_df = nn_tuning_results.reset_index(drop=True).copy()

plot_df["Experiment"] = range(1, len(plot_df)+1)

plt.figure(figsize=(14,6))

plt.plot(
    plot_df["Experiment"],
    plot_df["ROC_AUC"],
    marker="o",
    linewidth=2,
    label="ROC-AUC"
)

best_idx = plot_df["ROC_AUC"].idxmax()

plt.scatter(
    plot_df.loc[best_idx, "Experiment"],
    plot_df.loc[best_idx, "ROC_AUC"],
    s=200,
    label="Best ROC-AUC"
)

plt.title(
    "Neural Network ROC-AUC Progression",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Experiment Number", fontsize=13)
plt.ylabel("ROC-AUC", fontsize=13)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

VALIDATION LOSS vs ARCHITECTURE

In [ ]:
metrics = [
    "Accuracy",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC"
]

plt.figure(figsize=(16,7))

for metric in metrics:

    plt.plot(
        plot_df["Experiment"],
        plot_df[metric],
        marker="o",
        linewidth=2,
        label=metric
    )

plt.title(
    "Neural Network Metrics Progression Across Architectures",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Experiment Number", fontsize=13)
plt.ylabel("Score", fontsize=13)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

VALIDATION LOSS vs ARCHITECTURE

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    plot_df["Experiment"],
    plot_df["Best_Val_Loss"],
    marker="o",
    linewidth=2,
    label="Validation Loss"
)

best_loss_idx = plot_df["Best_Val_Loss"].idxmin()

plt.scatter(
    plot_df.loc[best_loss_idx, "Experiment"],
    plot_df.loc[best_loss_idx, "Best_Val_Loss"],
    s=200,
    label="Lowest Validation Loss"
)

plt.title(
    "Validation Loss Progression Across Architectures",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Experiment Number", fontsize=13)
plt.ylabel("Validation Loss", fontsize=13)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

RECALL vs ROC-AUC TRADEOFF

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=plot_df,
    x="Recall",
    y="ROC_AUC",
    size="PR_AUC",
    hue="F1",
    sizes=(100,400),
    palette="viridis"
)

plt.title(
    "Recall vs ROC-AUC Tradeoff",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Recall / Sensitivity", fontsize=13)
plt.ylabel("ROC-AUC", fontsize=13)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

The Recall vs ROC-AUC tradeoff plot demonstrates the balance between mortality detection capability and overall classification performance across different neural network architectures. Models located toward the upper-right region of the graph achieved the best compromise, combining higher sensitivity with stronger discrimination between survivors and non-survivors. Larger marker sizes in this region also indicate improved PR-AUC performance, suggesting better prediction quality for the minority mortality class.

The graph further shows that some architectures achieved very high recall but lower ROC-AUC, indicating increased mortality detection at the expense of more false positives, while others achieved strong ROC-AUC but lower sensitivity, potentially missing critically ill patients. From a clinical perspective, the most desirable models are those maintaining both relatively high recall and high ROC-AUC, since early identification of high-risk septic patients is more important than maximizing overall accuracy alone.

#Evaluating tuned NN on validation set

In [ ]:
best_model.eval()

with torch.no_grad():
    tuned_val_logits = best_model(X_val_tensor).squeeze()
    tuned_val_probs = torch.sigmoid(tuned_val_logits).cpu().numpy()

y_val_pred_nn_tuned = (tuned_val_probs >= 0.5).astype(int)
y_val_proba_nn_tuned = tuned_val_probs

In [ ]:
nn_tuned_results = evaluate_classifier(
    y_val,
    y_val_pred_nn_tuned,
    y_val_proba_nn_tuned,
    model_name="Tuned Neural Network"
)

In [ ]:


sns.set_style("whitegrid")

# CONFUSION MATRIX

cm = confusion_matrix(y_val, y_val_pred_nn_tuned)

plt.figure(figsize=(7,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="rocket",
    linewidths=2,
    linecolor="white",
    cbar=True,
    annot_kws={"fontsize":18, "fontweight":"bold"}
)

plt.title(
    "Tuned Neural Network — Confusion Matrix",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Label",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Label",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    [0.5, 1.5],
    ["Predicted Survived", "Predicted Died"],
    fontsize=12
)

plt.yticks(
    [0.5, 1.5],
    ["Actual Survived", "Actual Died"],
    fontsize=12,
    rotation=0
)

plt.tight_layout()
plt.show()


In [ ]:

# ROC CURVE

fpr, tpr, _ = roc_curve(
    y_val,
    y_val_proba_nn_tuned
)

auc_score = roc_auc_score(
    y_val,
    y_val_proba_nn_tuned
)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    linewidth=3,
    label=f"ROC-AUC = {auc_score:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    linewidth=2
)

plt.title(
    "Tuned Neural Network — ROC Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "False Positive Rate",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "True Positive Rate (Sensitivity)",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# PRECISION-RECALL CURVE

precision, recall, _ = precision_recall_curve(
    y_val,
    y_val_proba_nn_tuned
)

pr_auc = average_precision_score(
    y_val,
    y_val_proba_nn_tuned
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    linewidth=3,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.title(
    "Tuned Neural Network — Precision-Recall Curve",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel(
    "Recall",
    fontsize=14,
    fontweight="bold"
)

plt.ylabel(
    "Precision",
    fontsize=14,
    fontweight="bold"
)

plt.legend(fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# PERFORMANCE METRICS BARPLOT

metrics_dict = {

    "Accuracy": 0.606125,
    "Precision": 0.433276,
    "Recall": 0.844595,
    "Specificity": 0.497696,
    "F1-score": 0.572738,
    "ROC-AUC": 0.737120,
    "PR-AUC": 0.583186
}

metrics_df = pd.DataFrame({

    "Metric": list(metrics_dict.keys()),
    "Score": list(metrics_dict.values())

})

plt.figure(figsize=(10,6))

ax = sns.barplot(
    data=metrics_df,
    x="Metric",
    y="Score",
    palette="mako"
)

plt.ylim(0,1)

plt.title(
    "Tuned Neural Network — Performance Metrics",
    fontsize=20,
    fontweight="bold"
)

plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("")

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.xticks(rotation=20, fontsize=11)

sns.despine()

plt.tight_layout()
plt.show()


# BASELINE vs TUNED NEURAL NETWORK COMPARISON

In [ ]:


nn_compare_df = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1-score",
        "ROC-AUC",
        "PR-AUC"
    ],

    "Baseline NN": [
        0.700106,
        0.518519,
        0.567568,
        0.760369,
        0.541935,
        0.757608,
        0.605921
    ],

    "Tuned NN": [
        0.606125,
        0.433276,
        0.844595,
        0.497696,
        0.572738,
        0.737120,
        0.583186
    ]
})

nn_compare_melted = nn_compare_df.melt(
    id_vars="Metric",
    var_name="Model",
    value_name="Score"
)

plt.figure(figsize=(14,6))

ax = sns.barplot(
    data=nn_compare_melted,
    x="Metric",
    y="Score",
    hue="Model",
    palette="viridis"
)

plt.ylim(0,1)

plt.title(
    "Baseline vs Tuned Neural Network Performance",
    fontsize=20,
    fontweight="bold"
)

plt.xlabel("")
plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    rotation=20,
    fontsize=11
)

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

plt.legend(
    title="Model",
    fontsize=11
)

sns.despine()

plt.tight_layout()

plt.show()

The comparison between the baseline and tuned neural network models reveals a major shift in prediction behavior after hyperparameter tuning. The tuned neural network significantly improved recall/sensitivity from approximately 57% to 84%, meaning the model became substantially better at detecting mortality cases. Clinically, this is highly important because fewer critically ill septic patients are missed.

However, this improvement came at the expense of reduced precision, specificity, and overall accuracy. The tuned model generated more false positive mortality predictions, indicating a more aggressive screening behavior. Although the ROC-AUC slightly decreased, the tuned model achieved a higher F1-score, reflecting a better balance between precision and recall under an imbalanced clinical setting.

From a clinical perspective, the tuned neural network behaves like a high-sensitivity early warning system, prioritizing patient safety and mortality detection over strict classification accuracy.


# FINAL MODEL COMPARISON
# Tuned Logistic Regression vs Tuned SVM vs Tuned NN


In [ ]:

sns.set_style("whitegrid")

final_models_df = pd.DataFrame({

    "Metric": [

        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1-score",
        "ROC-AUC",
        "PR-AUC"
    ],

    "Tuned Logistic Regression": [

        0.740232,
        0.583333,
        0.591216,
        0.807988,
        0.587248,
        0.768599,
        0.627698
    ],

    "Tuned SVM": [

        0.687434,
        0.500000,
        0.003378,
        0.998464,
        0.006711,
        0.574592,
        0.340573
    ],

    "Tuned Neural Network": [

        0.606125,
        0.433276,
        0.844595,
        0.497696,
        0.572738,
        0.737120,
        0.583186
    ]
})

final_models_melted = final_models_df.melt(

    id_vars="Metric",

    var_name="Model",

    value_name="Score"
)

plt.figure(figsize=(16,7))

ax = sns.barplot(

    data=final_models_melted,

    x="Metric",

    y="Score",

    hue="Model",

    palette="viridis"
)

plt.ylim(0,1)

plt.title(
    "Final Comparison of Tuned Models",
    fontsize=22,
    fontweight="bold"
)

plt.xlabel("")
plt.ylabel(
    "Score",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(
    rotation=20,
    fontsize=12
)

for p in ax.patches:

    height = p.get_height()

    ax.annotate(
        f"{height:.2f}",
        (p.get_x() + p.get_width()/2, height),
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

plt.legend(
    title="Model",
    fontsize=11
)

sns.despine()

plt.tight_layout()

plt.show()

RADAR CHART

In [ ]:

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1-score",
    "ROC-AUC",
    "PR-AUC"
]

logreg_scores = [
    0.740232,
    0.583333,
    0.591216,
    0.807988,
    0.587248,
    0.768599,
    0.627698
]

svm_scores = [
    0.687434,
    0.500000,
    0.003378,
    0.998464,
    0.006711,
    0.574592,
    0.340573
]

nn_scores = [
    0.606125,
    0.433276,
    0.844595,
    0.497696,
    0.572738,
    0.737120,
    0.583186
]

angles = np.linspace(
    0,
    2*np.pi,
    len(metrics),
    endpoint=False
).tolist()

angles += angles[:1]

logreg_scores += logreg_scores[:1]
svm_scores += svm_scores[:1]
nn_scores += nn_scores[:1]

fig, ax = plt.subplots(
    figsize=(8,8),
    subplot_kw=dict(polar=True)
)

ax.plot(
    angles,
    logreg_scores,
    linewidth=2,
    label="Logistic Regression"
)

ax.fill(
    angles,
    logreg_scores,
    alpha=0.1
)

ax.plot(
    angles,
    svm_scores,
    linewidth=2,
    label="SVM"
)

ax.fill(
    angles,
    svm_scores,
    alpha=0.1
)

ax.plot(
    angles,
    nn_scores,
    linewidth=2,
    label="Neural Network"
)

ax.fill(
    angles,
    nn_scores,
    alpha=0.1
)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=11)

plt.title(
    "Radar Comparison of Final Tuned Models",
    fontsize=18,
    fontweight="bold",
    pad=20
)

plt.legend(
    loc="upper right",
    bbox_to_anchor=(1.3,1.1)
)

plt.tight_layout()

plt.show()

The tuned Logistic Regression model appears to provide the best balance between discrimination, generalization, and clinical interpretability for early sepsis mortality prediction. Although the tuned Neural Network achieved substantially higher sensitivity, it did so at the expense of specificity and precision, generating a larger number of false positive mortality predictions. Meanwhile, the tuned SVM demonstrated severe generalization failure and was therefore unsuitable for deployment in this clinical prediction setting.

#FINAL SCIENTIFIC CONCLUSION

Among all evaluated models, the tuned Logistic Regression achieved the best overall balance between discrimination, sensitivity, specificity, and precision, while maintaining strong interpretability and stable generalization performance. Although the tuned Neural Network achieved substantially higher mortality sensitivity, it generated more false positive predictions, reducing specificity and overall reliability. Therefore, the tuned Logistic Regression model was selected as the final optimal model for independent test evaluation.

#independent test evaluation

1. Rebuild Final Tuned Logistic Regression

In [ ]:
final_logreg = LogisticRegression(

    C=10,

    penalty="l2",

    solver="liblinear",

    class_weight=None,

    max_iter=5000,

    random_state=42
)

2. Train on FULL TRAINING DATA

In [ ]:

# Combine train + validation
X_final_train = pd.concat([
    X_train_smote_selected,
    X_val_selected
])

y_final_train = pd.concat([
    y_train_smote,
    y_val
])

In [ ]:
final_logreg.fit(
    X_final_train,
    y_final_train
)

In [ ]:
# Predictions
y_test_pred = final_logreg.predict(
    X_test_selected
)

# Probabilities
y_test_proba = final_logreg.predict_proba(
    X_test_selected
)[:,1]

FINAL TEST EVALUATION

In [ ]:
final_test_results = evaluate_classifier(

    y_test,

    y_test_pred,

    y_test_proba,

    model_name="Final Logistic Regression Test Performance"
)

final_test_results

In [ ]:
plot_confusion_matrix(

    y_test,

    y_test_pred,

    title="Final Logistic Regression — Test Confusion Matrix"
)

In [ ]:
plot_roc_curve(

    y_test,

    y_test_proba,

    title="Final Logistic Regression — Test ROC Curve"
)

In [ ]:


precision, recall, _ = precision_recall_curve(
    y_test,
    y_test_proba
)

pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

plt.figure(figsize=(8,6))

plt.plot(
    recall,
    precision,
    linewidth=3,
    label=f"PR-AUC = {pr_auc:.3f}"
)

plt.title(
    "Final Logistic Regression — Test PR Curve",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()

The final tuned Logistic Regression model demonstrated strong and stable generalization performance on the completely unseen test dataset, confirming its suitability for early sepsis mortality prediction. The model achieved a ROC-AUC of approximately 0.80 and a PR-AUC of approximately 0.66, indicating good discrimination between survivors and non-survivors while maintaining reliable performance on the minority mortality class.

The confusion matrix shows that the model correctly identified 188 mortality cases while correctly classifying 542 survivors. The recall/sensitivity of approximately 64% indicates that the model successfully detected a substantial proportion of high-risk septic patients, while the specificity of approximately 83% demonstrates good control of false positive mortality predictions. This balanced tradeoff between sensitivity and specificity is clinically important because it enables meaningful mortality detection without overwhelming clinicians with excessive false alarms.

Compared to the Neural Network and SVM models, the tuned Logistic Regression achieved the best overall balance between discrimination capability, stability, interpretability, and clinical reliability. The Neural Network achieved higher sensitivity but generated substantially more false positives, whereas the SVM failed to generalize appropriately. These findings suggest that mortality-related physiological patterns in the dataset are sufficiently captured through relatively stable linear relationships among key biomarkers such as lactate, SOFA score, renal dysfunction indicators, blood pressure abnormalities, and vasopressor usage.

Overall, the results demonstrate that a carefully preprocessed and properly tuned Logistic Regression model can provide robust and clinically meaningful early mortality prediction for septic ICU patients, while maintaining interpretability and strong generalization performance on unseen patient populations.